# Práctico Integrador – Pipeline de Datos de Noticias

Construir un pipeline en Python que procese datos de noticias provenientes de:

* Fuente A: texto no estructurado (TXT)
* Fuente B: datos estructurados (CSV)

El objetivo es comparar ambos escenarios y generar datasets limpios y consistentes.

**Contexto**

Una empresa recibe noticias desde múltiples fuentes.
Los datos presentan problemas de:

formatos inconsistentes
errores de carga
duplicados
datos faltantes

Tarea:

- Procesar ambas fuentes
- Limpiar y validar los datos
- Generar salidas estructuradas

Usá esta notebook para **explorar y construir el pipeline paso a paso**. Luego deberás llevar la solución a un **script (.py)**.


##  PARTE A:  Ingesta
Entrada

Archivo: noticias_crudas.txt

Leer archivo
Manejar:
* FileNotFoundError
* UnicodeDecodeError
* Registrar eventos en pipeline.log

In [ ]:
import re
import csv
import json
import logging
from datetime import datetime

# Configurar logging: todos los eventos del pipeline quedan registrados en pipeline.log
logging.basicConfig(
    filename='pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Leer el TXT manejando los errores más comunes de ingesta
lineas_crudas = []
try:
    with open('noticias_crudas.txt', encoding='utf-8') as f:
        lineas_crudas = f.readlines()
    logging.info(f"Archivo TXT leído correctamente: {len(lineas_crudas)} líneas")
    print(f"Líneas leídas del TXT: {len(lineas_crudas)}")
except FileNotFoundError:
    logging.error("noticias_crudas.txt no encontrado")
    print("Error: archivo no encontrado")
except UnicodeDecodeError:
    logging.error("Error de codificación en noticias_crudas.txt")
    print("Error: problema de codificación")


##  Parseo (OBLIGATORIO usar regex)

Extraer:

1. título
2. URL
3. fecha
4. categoría

In [ ]:
# PARSEO CON REGEX
# Cada línea del TXT tiene este formato:
#   Titulo: ... | Link: ... | Fecha: ... | Categoria: ...
#
# Estrategia: usamos re.search() con grupos de captura (.+?) para cada campo.
# El modificador ? hace la captura "no greedy" → se detiene en el primer | en lugar
# de llegar hasta el último, lo que evita capturar campos de más.
# (?:\||$) significa: "hasta el próximo | O hasta el fin de línea"

def parsear_linea_txt(linea):
    """Extrae titulo, url, fecha y categoria de una línea con formato fijo."""

    titulo_m = re.search(r'Titulo:\s*(.+?)\s*(?:\||$)', linea)
    link_m   = re.search(r'Link:\s*(.*?)\s*(?:\||$)', linea)
    fecha_m  = re.search(r'Fecha:\s*(.*?)\s*(?:\||$)', linea)
    # Categoria es el último campo: puede terminar en \n o fin de cadena
    cat_m    = re.search(r'Categoria:\s*(.*?)(?:\n|$)', linea)

    return {
        'titulo':    titulo_m.group(1).strip() if titulo_m else '',
        'url':       link_m.group(1).strip()   if link_m   else '',
        'fecha':     fecha_m.group(1).strip()  if fecha_m  else '',
        'categoria': cat_m.group(1).strip()    if cat_m    else '',
    }

# Aplicar el parseo a cada línea no vacía del archivo
noticias_txt_raw = []
for linea in lineas_crudas:
    if linea.strip():
        noticias_txt_raw.append(parsear_linea_txt(linea))

print(f"Noticias parseadas del TXT: {len(noticias_txt_raw)}\n")
for n in noticias_txt_raw:
    print(n)


## Limpieza y Normalización

- Título: formato legible (strip, capitalización)
- URL: debe comenzar con http:// o https://
- Fecha: normalizar a YYYY-MM-DD (usar strptime y strftime de datetime)
- Categoría: minúsculas; si falta → "sin_categoria"

In [ ]:
# FUNCIONES DE NORMALIZACIÓN
# Se definen aquí para reutilizarlas en ambas fuentes (TXT y CSV)

FORMATOS_FECHA = ['%Y-%m-%d', '%d/%m/%Y', '%Y/%m/%d']

def normalizar_titulo(titulo):
    # re.sub(r'\s+', ' ', ...) colapsa múltiples espacios en uno solo
    titulo = re.sub(r'\s+', ' ', titulo).strip()
    return titulo.capitalize()

def normalizar_url(url):
    url = url.strip()
    # Si no arranca con http:// o https:// (ignorando mayúsculas) se lo agregamos.
    # re.IGNORECASE permite matchear HTTP://, HTTPS://, etc.
    if url and not re.match(r'^https?://', url, re.IGNORECASE):
        url = 'http://' + url
    return url.lower()

def normalizar_fecha(fecha_str):
    """Intenta parsear la fecha con múltiples formatos. Devuelve 'YYYY-MM-DD' o None."""
    fecha_str = fecha_str.strip()
    for fmt in FORMATOS_FECHA:
        try:
            dt = datetime.strptime(fecha_str, fmt)
            return dt.strftime('%Y-%m-%d')
        except ValueError:
            continue
    return None  # ningún formato coincidió → fecha inválida

def normalizar_categoria(cat):
    cat = cat.strip().lower()
    return cat if cat else 'sin_categoria'

def normalizar_noticia(noticia):
    """Aplica todas las normalizaciones a un diccionario de noticia."""
    return {
        'titulo':    normalizar_titulo(noticia['titulo']),
        'url':       normalizar_url(noticia['url']),
        'fecha':     normalizar_fecha(noticia['fecha']),
        'categoria': normalizar_categoria(noticia['categoria']),
    }

print("Funciones de normalización definidas.")


## Limpieza

In [ ]:
# Aplicar normalización a las noticias crudas del TXT
noticias_txt_norm = [normalizar_noticia(n) for n in noticias_txt_raw]

print("Noticias TXT normalizadas:\n")
for n in noticias_txt_norm:
    print(n)


## Validación

- título no vacío
- URL válida
- fecha válida

## Duplicados
No puede haber dos noticias con la misma URL
Conservar la primera. Usar un conjunto

In [ ]:
# VALIDACIÓN Y DEDUPLICACIÓN

def es_url_valida(url):
    # Debe comenzar con http:// o https:// y NO ser un email (no contener @)
    return bool(re.match(r'^https?://', url)) and '@' not in url

def validar_noticia(noticia):
    """Devuelve (True, 'ok') si pasa todas las reglas, o (False, motivo) si no."""
    if not noticia['titulo']:
        return False, 'titulo vacio'
    if not es_url_valida(noticia['url']):
        return False, f"url invalida: {noticia['url']}"
    if noticia['fecha'] is None:
        return False, 'fecha invalida'
    return True, 'ok'

def filtrar_y_deduplicar(noticias, fuente=''):
    """
    Valida cada noticia y elimina duplicados por URL.
    Usa un set (urls_vistas) para detectar duplicados en O(1).
    Conserva la primera aparición de cada URL.
    """
    urls_vistas = set()
    validas     = []
    descartadas = []

    for n in noticias:
        ok, motivo = validar_noticia(n)
        if not ok:
            logging.warning(f"[{fuente}] Descartada '{n['titulo']}': {motivo}")
            descartadas.append({**n, 'motivo': motivo})
            continue
        if n['url'] in urls_vistas:
            logging.warning(f"[{fuente}] Duplicado descartado: {n['url']}")
            descartadas.append({**n, 'motivo': 'duplicado'})
            continue
        urls_vistas.add(n['url'])
        validas.append(n)

    return validas, descartadas

noticias_txt_validas, noticias_txt_desc = filtrar_y_deduplicar(noticias_txt_norm, 'TXT')
print(f"TXT → válidas: {len(noticias_txt_validas)} | descartadas: {len(noticias_txt_desc)}")
print("\nDescartadas TXT:")
for d in noticias_txt_desc:
    print(f"  {d['titulo']!r} → {d['motivo']}")


## PARTE B - Datos Estructurados (CSV)
Entrada

Archivo: noticias.csv
Leer usando módulo csv.

Limpiar, normalizar y validar siguiendo las mismas reglas que en la parte 1

## Pipeline

In [ ]:
# PARTE B - Pipeline CSV
# csv.DictReader usa la primera fila como nombres de columna y devuelve
# cada fila como un diccionario → mismo formato que usamos en la Parte A.

noticias_csv_raw = []
try:
    with open('noticias.csv', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for fila in reader:
            noticias_csv_raw.append({
                'titulo':    fila.get('titulo', '').strip(),
                'url':       fila.get('url', '').strip(),
                'fecha':     fila.get('fecha', '').strip(),
                'categoria': fila.get('categoria', '').strip(),
            })
    logging.info(f"CSV leído: {len(noticias_csv_raw)} filas")
    print(f"Filas leídas del CSV: {len(noticias_csv_raw)}")
except FileNotFoundError:
    logging.error("noticias.csv no encontrado")
    print("Error: archivo no encontrado")

# Mismo pipeline que en la Parte A: normalizar → validar → deduplicar
noticias_csv_norm    = [normalizar_noticia(n) for n in noticias_csv_raw]
noticias_csv_validas, noticias_csv_desc = filtrar_y_deduplicar(noticias_csv_norm, 'CSV')

print(f"\nCSV → válidas: {len(noticias_csv_validas)} | descartadas: {len(noticias_csv_desc)}")
print("\nDescartadas CSV:")
for d in noticias_csv_desc:
    print(f"  {d['titulo']!r} → {d['motivo']}")


##  PARTE C – Integración
Unificar ambas fuentes en una única lista
Mantener consistencia de formato

## Exportación

Generar:

✔ CSV final

noticias_limpias.csv

✔ JSON final

noticias_limpias.json

In [ ]:
# PARTE C - Integración y Exportación

# Unificar ambas fuentes en una sola lista
todas_noticias = noticias_txt_validas + noticias_csv_validas

# Segunda pasada de deduplicación: puede haber la misma URL en TXT y CSV
todas_noticias, extra_desc = filtrar_y_deduplicar(todas_noticias, 'MERGE')
print(f"Total noticias limpias tras merge: {len(todas_noticias)}")

# --- Exportar CSV ---
CAMPOS = ['titulo', 'url', 'fecha', 'categoria']
with open('noticias_limpias.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=CAMPOS)
    writer.writeheader()
    writer.writerows(todas_noticias)
print("Exportado: noticias_limpias.csv")

# --- Exportar JSON ---
# ensure_ascii=False conserva tildes y caracteres especiales sin escapar
with open('noticias_limpias.json', 'w', encoding='utf-8') as f:
    json.dump(todas_noticias, f, ensure_ascii=False, indent=2)
print("Exportado: noticias_limpias.json")


## PARTE E – Métricas

Mostrar:

- total procesados
- válidos
- descartados
- % de noticias por categoría "tecnologia"

In [ ]:
# PARTE E - Métricas del pipeline

total_txt = len(noticias_txt_raw)
total_csv = len(noticias_csv_raw)
total_procesados = total_txt + total_csv

total_validas    = len(todas_noticias)
total_desc_txt   = len(noticias_txt_desc)
total_desc_csv   = len(noticias_csv_desc)
total_desc_merge = len(extra_desc)
total_descartados = total_desc_txt + total_desc_csv + total_desc_merge

tec_count = sum(1 for n in todas_noticias if n['categoria'] == 'tecnologia')
pct_tec   = (tec_count / total_validas * 100) if total_validas else 0

print("=" * 40)
print("       MÉTRICAS DEL PIPELINE")
print("=" * 40)
print(f"  Total procesados  : {total_procesados}")
print(f"    - desde TXT     : {total_txt}")
print(f"    - desde CSV     : {total_csv}")
print(f"  Válidas (finales) : {total_validas}")
print(f"  Descartadas total : {total_descartados}")
print(f"    - por TXT       : {total_desc_txt}")
print(f"    - por CSV       : {total_desc_csv}")
print(f"    - por duplicado entre fuentes: {total_desc_merge}")
print(f"  % tecnología      : {pct_tec:.1f}%")
print("=" * 40)
